# 1. Importing libraries and loading datasets

In [1]:
import numpy as np
import pandas as pd

# Encoders
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import LabelEncoder

# Modelling
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# Random Forest
from sklearn.ensemble import RandomForestClassifier

# XGBoost
from xgboost import XGBClassifier

# LightGBM
from lightgbm import LGBMClassifier

# Voting Classifier
from sklearn.ensemble import VotingClassifier

In [2]:
train_data = pd.read_csv('../input/titanic/train.csv')
test_data = pd.read_csv('../input/titanic/test.csv')

# 2. Explore data

In [3]:
train_data

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [4]:
train_data.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [5]:
print("Columns: \n{0} ".format(train_data.columns.tolist()))

Columns: 
['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'] 


# 3. Basic data check

## Missing values

In [6]:
missing_values = train_data.isna().any()
print('Columns which have missing values: \n{0}'.format(missing_values[missing_values == True].index.tolist()))

Columns which have missing values: 
['Age', 'Cabin', 'Embarked']


In [7]:
print("Percentage of missing values in `Age` column: {0:.2f}".format(100.*(train_data.Age.isna().sum()/len(train_data))))
print("Percentage of missing values in `Cabin` column: {0:.2f}".format(100.*(train_data.Cabin.isna().sum()/len(train_data))))
print("Percentage of missing values in `Embarked` column: {0:.2f}".format(100.*(train_data.Embarked.isna().sum()/len(train_data))))

Percentage of missing values in `Age` column: 19.87
Percentage of missing values in `Cabin` column: 77.10
Percentage of missing values in `Embarked` column: 0.22


## Check for duplicates

In [8]:
duplicates = train_data.duplicated().sum()
print('Duplicates in train data: {0}'.format(duplicates))

Duplicates in train data: 0


## Categorical variables

In [9]:
categorical = train_data.nunique().sort_values(ascending=True)
print('Categorical variables in train data: \n{0}'.format(categorical))

Categorical variables in train data: 
Survived         2
Sex              2
Pclass           3
Embarked         3
SibSp            7
Parch            7
Age             88
Cabin          147
Fare           248
Ticket         681
PassengerId    891
Name           891
dtype: int64


# 4. Data cleaning

In [10]:
for data in [train_data, test_data]:
    # Too many missing values
    data.drop(['Cabin'], axis=1, inplace=True)
    # Probably will not provide some useful information
    data.drop(['Ticket', 'Fare'], axis=1, inplace=True)

In [11]:
train_data.tail()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Embarked
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,C
890,891,0,3,"Dooley, Mr. Patrick",male,32.0,0,0,Q


# 5. Feature engineering

In [12]:
# Find the women and boys
for data in [train_data, test_data]:
    data['Title'] = data.Name.str.split(',').str[1].str.split('.').str[0].str.strip()
    data['Woman_Or_Boy'] = (data.Title == 'Master') | (data.Sex == 'female')
    data.drop('Title', axis=1, inplace=True)
    data.drop('Name', axis=1, inplace=True)

In [13]:
# Encode 'Sex' and 'Woman_Or_Boy' columns
label_encoder = LabelEncoder()
for data in [train_data, test_data]:
    data['Sex'] = label_encoder.fit_transform(data['Sex'])
    data['Woman_Or_Boy'] = label_encoder.fit_transform(data['Woman_Or_Boy'])

In [14]:
# Merge two data to get the average Age and fill the column
all_data = pd.concat([train_data, test_data])
average = all_data.Age.median()
print("Average Age: {0}".format(average))
for data in [train_data, test_data]:
    data.fillna(value={'Age': average}, inplace=True)

Average Age: 28.0


In [15]:
# Get the most common Embark and fill the column
most_common = all_data.Embarked.mode()
print("Most common Embarked value: {0}".format(most_common[0]))
for data in [train_data, test_data]:
    data.fillna(value={'Embarked': most_common[0]}, inplace=True)

Most common Embarked value: S


In [16]:
# Create categorical variable for traveling alone
# Credits to https://www.kaggle.com/vaishvik25/titanic-eda-fe-3-model-decision-tree-viz
for data in [train_data, test_data]:
    data['TravelAlone'] = np.where(data["SibSp"] + data["Parch"] > 0, 0, 1)
    data.drop('SibSp', axis=1, inplace=True)
    data.drop('Parch', axis=1, inplace=True)

In [17]:
# Encode 'Embarked' column
one_hot_encoder = OneHotEncoder(sparse=False)
def encode_embarked(data):
    encoded = pd.DataFrame(one_hot_encoder.fit_transform(data[['Embarked']]))
    encoded.columns = one_hot_encoder.get_feature_names(['Embarked'])
    data.drop(['Embarked'], axis=1, inplace=True)
    data = data.join(encoded)
    return data
train_data = encode_embarked(train_data)
test_data = encode_embarked(test_data)

In [18]:
train_data.tail()

,PassengerId,Survived,Pclass,Sex,Age,Woman_Or_Boy,TravelAlone,Embarked_C,Embarked_Q,Embarked_S
886,887,0,2,1,27.0,0,1,0.0,0.0,1.0
887,888,1,1,0,19.0,1,1,0.0,0.0,1.0
888,889,0,3,0,28.0,1,0,0.0,0.0,1.0
889,890,1,1,1,26.0,0,1,1.0,0.0,0.0
890,891,0,3,1,32.0,0,1,0.0,1.0,0.0


# 6. Modelling

Use all top scoring models to test them for GridSearch VotingClassifier.  
https://www.kaggle.com/sfktrkl/titanic-hyperparameter-tuning-gridsearchcv

In [19]:
# Set X and y
X = train_data.drop(['Survived', 'PassengerId'], axis=1)
y = train_data['Survived']
test_X = test_data.drop(['PassengerId'], axis=1)

In [20]:
# To store models created
best_models = {}

# Split data
train_X, val_X, train_y, val_y = train_test_split(X, y, random_state=1)

def print_best_parameters(hyperparameters, best_parameters):
    value = "Best parameters: "
    for key in hyperparameters:
        value += str(key) + ": " + str(best_parameters[key]) + ", "
    if hyperparameters:
        print(value[:-2])

def get_best_model(estimator, hyperparameters):
    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
    grid_search = GridSearchCV(estimator=estimator, param_grid=hyperparameters,
                               n_jobs=-1, cv=cv, scoring="accuracy")
    best_model = grid_search.fit(train_X, train_y)
    best_parameters = best_model.best_estimator_.get_params()
    print_best_parameters(hyperparameters, best_parameters)
    return best_model

def evaluate_model(model, name):
    print("Accuracy score:", accuracy_score(train_y, model.predict(train_X)))
    best_models[name] = model

In [21]:
print("Features: \n{0} ".format(X.columns.tolist()))

Features: 
['Pclass', 'Sex', 'Age', 'Woman_Or_Boy', 'TravelAlone', 'Embarked_C', 'Embarked_Q', 'Embarked_S'] 


## Classifiers

In [22]:
# I couldn't find a way to set fit_params of XGBClasssifier through GridSearchCV, so did a little trick.
# https://stackoverflow.com/questions/35545733/how-do-you-use-fit-params-for-randomizedsearch-with-votingclassifier-in-sklearn
class MyXGBClassifier(XGBClassifier):
    def fit(self, X, y=None):
        return super(XGBClassifier, self).fit(X, y,
                                              verbose=False,
                                              early_stopping_rounds=40,
                                              eval_metric='logloss',
                                              eval_set=[(val_X, val_y)])

In [23]:
# Models from, https://www.kaggle.com/sfktrkl/titanic-hyperparameter-tuning-gridsearchcv
randomForest = RandomForestClassifier(random_state=1, n_estimators=20, max_features='auto',
                                      criterion='gini', max_depth=4, min_samples_split=2,
                                      min_samples_leaf=3)
xgbClassifier = MyXGBClassifier(seed=1, tree_method='gpu_hist', predictor='gpu_predictor',
                                use_label_encoder=False, learning_rate=0.4, gamma=0.4,
                                max_depth=4, reg_lambda=0, reg_alpha=0.1)
lgbmClassifier = LGBMClassifier(random_state=1, device='gpu', boosting_type='dart',
                                num_leaves=8, learning_rate=0.1, n_estimators=100,
                                reg_alpha=1, reg_lambda=1)

classifiers = [
    ('randomForest', randomForest),
    ('xgbClassifier', xgbClassifier),
    ('lgbmClassifier', lgbmClassifier)
]

## [VotingClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.VotingClassifier.html)

* **estimators: list of (str, estimator) tuples**  
    Invoking the fit method on the VotingClassifier will fit clones of those original estimators that will be stored in the class attribute self.estimators_. An estimator can be set to 'drop' using set_params.

* **voting: {‘hard’, ‘soft’}, default=’hard’**  
    If ‘hard’, uses predicted class labels for majority rule voting. Else if ‘soft’, predicts the class label based on the argmax of the sums of the predicted probabilities, which is recommended for an ensemble of well-calibrated classifiers.
    
* **weights: array-like of shape (n_classifiers,), default=None**  
    Sequence of weights (float or int) to weight the occurrences of predicted class labels (hard voting) or class probabilities before averaging (soft voting). Uses uniform weights if None.
    
* **n_jobs: int, default=None**  
    The number of jobs to run in parallel for fit. None means 1 unless in a joblib.parallel_backend context. -1 means using all processors. See Glossary for more details.

In [24]:
hyperparameters = {
    'n_jobs'  : [-1],
    'voting'  : ['hard', 'soft'],
    'weights' : [(1, 1, 1),
                (2, 1, 1), (1, 2, 1), (1, 1, 2),
                (2, 2, 1), (1, 2, 2), (2, 1, 2),
                (3, 2, 1), (1, 3, 2), (2, 1, 3), (3, 1, 2)]
}
estimator = VotingClassifier(estimators=classifiers)
best_model_voting = get_best_model(estimator, hyperparameters)

Best parameters: n_jobs: -1, voting: soft, weights: (3, 1, 2)


In [25]:
evaluate_model(best_model_voting.best_estimator_, 'voting')

Accuracy score: 0.842814371257485


# 7. Submission

In [26]:
# Get predictions for each model and create submission files
for model in best_models:
    predictions = best_models[model].predict(test_X)
    output = pd.DataFrame({'PassengerId': test_data.PassengerId, 'Survived': predictions})
    output.to_csv('submission_' + model + '.csv', index=False)